# Parent Atlas — EmbeddingGemma Poincaré hierarchy experiment

This Colab experiment freezes a version-pinned EmbeddingGemma text encoder and trains a small Poincaré-ball hierarchy adapter over explicitly asserted named subclass edges. It is OnT-inspired, not OnT, not an OWL reasoner, and not a fine-tune of Google's model. An optional role rotation/scale head trains only when separately supplied, source-identified role assertions pass the minimum-count gate; role quality remains unevaluated without a reserved relation test split. It exports advisory ranking metrics and separate experiment artifacts; it never writes to Parent Atlas stores or replaces `semantic_768`.

The repository contains a browser ONNX EmbeddingGemma artifact at `sveltekit-frontend/static/embeddinggemma_300m_onnx/model.onnx` (the similarly named `models/embeddinggemma_300m_onnx` directory has no ONNX graph). This experiment uses the PyTorch SentenceTransformer checkpoint from Hugging Face; ONNX inference cannot train this adapter and is not assumed recipe-compatible. Choose `google/embeddinggemma-300m` or, for a separate run, `google/embeddinggemma-2` (text-only config). Revisions are pinned in the config cell. Accept Google's model access terms before loading. Default settings run only an offline synthetic smoke; for a real advisory experiment, upload a reviewed OWL/CSV cohort, set `SOURCE_MODE` to `owl` or `csv`, and set `OFFLINE_SMOKE=False`.

In [ ]:
%pip -q install -U "sentence-transformers" "transformers" "huggingface_hub" "rdflib>=7,<8" "pandas>=2,<4" "psutil>=5.9,<8"

In [ ]:
import hashlib, io, json, random, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import psutil
from rdflib import Graph, RDFS, URIRef
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

MODEL_REVISIONS = {"google/embeddinggemma-300m": "57c266a740f537b4dc058e1b0cda161fd15afa75", "google/embeddinggemma-2": "914f7f89142e33e77833254d9c9b90c3cef7303b"}
MODEL_ID = "google/embeddinggemma-300m"  # choose embeddinggemma-2 only for a separate experiment
MODEL_REVISION = MODEL_REVISIONS[MODEL_ID]
SOURCE_MODE = "demo"  # demo | owl | csv; demo is synthetic and only valid for a smoke run
OFFLINE_SMOKE = True  # deterministic synthetic vectors; skips Hugging Face download/inference
SEED = 20261024
GEOMETRY_DIM = 64
EPOCHS, LEARNING_RATE = 120, 0.004
MIN_ASSERTIONS = 12  # below this, refuse to report held-out ranking metrics
def require_model_memory(model_id, device):
    required_gib = 6.0 if model_id == "google/embeddinggemma-2" else 4.0
    available_bytes = torch.cuda.mem_get_info()[0] if device == "cuda" else psutil.virtual_memory().available
    required_bytes = int(required_gib * (1024**3))
    if available_bytes < required_bytes:
        raise MemoryError(f"Model load refused before download: {available_bytes / (1024**3):.2f} GiB available, {required_gib:.1f} GiB required for {model_id}. Free memory or use a larger Colab runtime.")
    return {"device": device, "available_gib": round(available_bytes / (1024**3), 2), "required_gib": required_gib}
RUN_DIR = Path(f"/content/parent_atlas_poincare_{MODEL_ID.rsplit('/', 1)[-1]}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cpu"  # select CUDA only after input-size and split gates pass
DTYPE = torch.float32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print({"model": MODEL_ID, "device_preflight": DEVICE, "dtype_preflight": str(DTYPE), "gpu_initialization": "DEFERRED_UNTIL_DATA_GATES_PASS"})

## Input and evidence boundary

Upload one OWL/RDF/Turtle ontology, or use CSV mode with `classes.csv` (`id,label,description`), `subclass_edges.csv` (`child,parent,source_id,verified`), and optional `existential_axioms.csv` (`subject,role,filler,source_id,verified`). CSV `verified` is only an uploader assertion; rows without source IDs or `verified=true` are excluded. The notebook extracts asserted named `rdfs:subClassOf` edges only. Hashes bind bytes, not source authorization or production admission.

In [ ]:
from google.colab import files

def sha256_bytes(data): return hashlib.sha256(data).hexdigest()
def label_for(graph, iri):
    value = next(graph.objects(URIRef(iri), RDFS.label), None)
    return str(value) if value is not None else str(iri).rstrip("/#").rsplit("#", 1)[-1].rsplit("/", 1)[-1].replace("_", " ")
def verified_flag(value): return str(value).strip().casefold() in {"true", "1", "yes"}
INPUT_ARTIFACTS = []
edge_sources = {}
existence = pd.DataFrame(columns=["subject", "role", "filler", "source_id", "verified"])

if SOURCE_MODE == "demo":
    if not OFFLINE_SMOKE: raise ValueError("Synthetic demo data is allowed only with OFFLINE_SMOKE=True")
    ids = ["AtlasConceptRoot"] + [f"AtlasConcept{i:02d}" for i in range(20)]
    labels = {item: item.replace("AtlasConcept", "Atlas Concept ") for item in ids}
    descriptions = {item: "SYNTHETIC SMOKE FIXTURE ONLY" for item in ids}
    asserted = [(item, "AtlasConceptRoot") for item in ids if item != "AtlasConceptRoot"]
    INPUT_SHA256 = sha256_bytes(json.dumps({"ids":ids,"edges":asserted},sort_keys=True).encode())
    edge_sources = {edge: ["SYNTHETIC_DEMO"] for edge in asserted}
    INPUT_ARTIFACTS = [{"name": "SYNTHETIC_DEMO", "sha256": INPUT_SHA256}]
elif SOURCE_MODE == "owl":
    uploaded = files.upload()
    files_in = [name for name in uploaded if name.lower().endswith((".owl", ".rdf", ".ttl"))]
    if len(files_in) != 1: raise ValueError("Upload exactly one .owl, .rdf, or .ttl file")
    ontology_path = Path(files_in[0]); ontology_bytes = ontology_path.read_bytes()
    graph = Graph(); graph.parse(data=ontology_bytes, format={".ttl": "turtle", ".rdf": "xml", ".owl": "xml"}[ontology_path.suffix.lower()])
    asserted = sorted({(str(child), str(parent)) for child, parent in graph.subject_objects(RDFS.subClassOf) if isinstance(child, URIRef) and isinstance(parent, URIRef)})
    ids = sorted({item for edge in asserted for item in edge})
    labels = {item: label_for(graph, item) for item in ids}
    descriptions = {item: "" for item in ids}
    INPUT_SHA256 = sha256_bytes(ontology_bytes)
    edge_sources = {edge: [f"sha256:{INPUT_SHA256}"] for edge in asserted}
    INPUT_ARTIFACTS = [{"name": ontology_path.name, "sha256": INPUT_SHA256}]
elif SOURCE_MODE == "csv":
    uploaded = files.upload()
    if not {"classes.csv", "subclass_edges.csv"}.issubset(uploaded): raise ValueError("Upload classes.csv and subclass_edges.csv")
    unexpected = set(uploaded) - {"classes.csv", "subclass_edges.csv", "existential_axioms.csv"}
    if unexpected: raise ValueError(f"Unexpected CSV upload(s): {sorted(unexpected)}")
    classes_raw = pd.read_csv(io.BytesIO(uploaded["classes.csv"]), dtype=str).fillna("")
    edges_raw = pd.read_csv(io.BytesIO(uploaded["subclass_edges.csv"]), dtype=str).fillna("")
    if not {"id", "label"}.issubset(classes_raw) or not {"child", "parent", "source_id", "verified"}.issubset(edges_raw): raise ValueError("CSV columns are incomplete; edges require source_id and verified")
    classes_raw[["id", "label"]] = classes_raw[["id", "label"]].astype(str).apply(lambda col: col.str.strip())
    if classes_raw.id.duplicated().any(): raise ValueError("Duplicate class IDs")
    if (classes_raw.id == "").any() or (classes_raw.label == "").any(): raise ValueError("Empty class ID or label")
    ids = sorted(classes_raw.id.tolist()); labels = dict(zip(classes_raw.id, classes_raw.label))
    descriptions = dict(zip(classes_raw.id, classes_raw.get("description", pd.Series([""] * len(classes_raw)))))
    edges_raw = edges_raw[edges_raw.verified.map(verified_flag) & edges_raw.source_id.astype(str).str.strip().ne("")].copy()
    edges_raw[["child", "parent"]] = edges_raw[["child", "parent"]].astype(str).apply(lambda col: col.str.strip())
    edges_raw["source_id"] = edges_raw.source_id.astype(str).str.strip()
    asserted = sorted(set(zip(edges_raw.child, edges_raw.parent)))
    edge_sources = {edge: sorted(set(rows.source_id.astype(str))) for edge, rows in edges_raw.groupby(["child", "parent"], sort=True)}
    if "existential_axioms.csv" in uploaded:
        existence_raw = pd.read_csv(io.BytesIO(uploaded["existential_axioms.csv"]), dtype=str).fillna("")
        required = {"subject", "role", "filler", "source_id", "verified"}
        if not required.issubset(existence_raw): raise ValueError("existential_axioms.csv columns are incomplete")
        existence = existence_raw[existence_raw.verified.map(verified_flag) & existence_raw.source_id.astype(str).str.strip().ne("")].copy()
        existence = existence[["subject", "role", "filler", "source_id"]].astype(str).apply(lambda col: col.str.strip()).drop_duplicates().reset_index(drop=True)
    INPUT_SHA256 = sha256_bytes(b"\0".join(name.encode() + b"\0" + uploaded[name] for name in sorted(uploaded)))
    INPUT_ARTIFACTS = [{"name": name, "sha256": sha256_bytes(uploaded[name])} for name in sorted(uploaded)]
else: raise ValueError("SOURCE_MODE must be owl or csv")

if len(ids) < 3 or not asserted: raise ValueError("Ontology has too few named classes or asserted edges")
if any(child not in labels or parent not in labels or child == parent for child, parent in asserted): raise ValueError("Edge references an unknown class or self-loop")
if len(asserted) < MIN_ASSERTIONS: raise ValueError(f"Only {len(asserted)} asserted named edges; need {MIN_ASSERTIONS} for held-out evaluation. No training/evaluation artifact was produced.")
if len(existence) and any(row.subject not in set(ids) or row.filler not in set(ids) for row in existence.itertuples()): raise ValueError("Existential axiom references unknown class ID")
if SOURCE_MODE == "owl": existence = pd.DataFrame(columns=["subject", "role", "filler", "source_id"])
DATASET_SHA256 = sha256_bytes(json.dumps({"classes": sorted((item, labels[item], descriptions.get(item, "")) for item in ids), "edges": sorted((child, parent, edge_sources.get((child, parent), [])) for child, parent in asserted), "existential_axioms": sorted(tuple(row) for row in existence.astype(str).itertuples(index=False, name=None))}, sort_keys=True).encode())
print({"classes": len(ids), "asserted_edges": len(asserted), "input_sha256": INPUT_SHA256, "dataset_sha256": DATASET_SHA256, "role_axioms": len(existence), "authority": "SYNTHETIC_SMOKE" if SOURCE_MODE == "demo" else "CALLER_SUPPLIED_UNVERIFIED"})

In [ ]:
def transitive_closure(edges, nodes):
    parents = {node: set() for node in nodes}
    for child, parent in edges: parents[child].add(parent)
    state, closure = {}, set()
    def visit(node):
        if state.get(node) == 1: raise ValueError(f"Cycle detected at {node}")
        if state.get(node) == 2: return
        state[node] = 1
        for parent in parents[node]: visit(parent)
        state[node] = 2
    for node in nodes: visit(node)
    for node in nodes:
        stack = list(parents[node]); seen = set()
        while stack:
            parent = stack.pop()
            if parent not in seen:
                seen.add(parent); closure.add((node, parent)); stack.extend(parents[parent])
    return closure

full_closure = transitive_closure(asserted, ids)
rng = np.random.default_rng(SEED)
for attempt in range(256):
    order = rng.permutation(len(asserted))
    n_test = max(2, int(round(len(order) * 0.15))); n_val = max(2, int(round(len(order) * 0.15)))
    if len(order) - n_test - n_val < 2: continue
    test_edges = [asserted[i] for i in order[:n_test]]
    val_edges = [asserted[i] for i in order[n_test:n_test+n_val]]
    train_edges = [asserted[i] for i in order[n_test+n_val:]]
    train_closure = transitive_closure(train_edges, ids)
    if not (set(val_edges) | set(test_edges)) & train_closure: break
else: raise ValueError("No closure-leak-free split found; gather a larger or structurally suitable reviewed ontology")

assert not (set(train_edges) & set(val_edges) or set(train_edges) & set(test_edges) or set(val_edges) & set(test_edges))
print({"train": len(train_edges), "validation": len(val_edges), "test": len(test_edges), "full_closure_pairs": len(full_closure), "closure_leak_free": True})

## Frozen EmbeddingGemma encoder

The encoder is loaded at a fixed Hugging Face commit and set to evaluation mode. This experiment uses a symmetric sentence-similarity prefix, not the production query/document recipe and not `semantic_768` admission. The small ontology adapter below is the only trainable component.

In [ ]:
if not OFFLINE_SMOKE:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
    MODEL_MEMORY_PREFLIGHT = require_model_memory(MODEL_ID, DEVICE)
    print({"model_memory_preflight": MODEL_MEMORY_PREFLIGHT})
    if DEVICE == "cuda": torch.cuda.manual_seed_all(SEED)
if OFFLINE_SMOKE:
    loaded_revision = "NOT_LOADED_SYNTHETIC"
    def smoke_vector(text):
        seed = int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16)
        value = np.random.default_rng(seed).normal(size=768).astype(np.float32)
        return value / np.linalg.norm(value)
    vectors = np.stack([smoke_vector(labels[item]) for item in ids])
else:
    if SOURCE_MODE == "demo": raise ValueError("Real model execution requires owl/csv input, not synthetic demo data")
    if len(MODEL_REVISION) != 40 or any(ch not in "0123456789abcdef" for ch in MODEL_REVISION): raise ValueError("MODEL_REVISION must be an immutable Hub commit SHA")
    hub_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
    if hub_revision != MODEL_REVISION: raise ValueError("Hub did not resolve the requested model revision")
    model_kwargs = {"revision": MODEL_REVISION, "device": DEVICE, "model_kwargs": {"torch_dtype": DTYPE}}
    if MODEL_ID == "google/embeddinggemma-2": model_kwargs["config_kwargs"] = {"vision_config": None, "audio_config": None}
    encoder = SentenceTransformer(MODEL_ID, **model_kwargs)
    encoder.eval()
    loaded_revision = getattr(getattr(getattr(encoder[0], "auto_model", None), "config", None), "_commit_hash", None)
    if loaded_revision != MODEL_REVISION: raise ValueError("Loaded model revision differs from the pinned revision")
    text_by_id = {item: (labels[item] + (". " + descriptions.get(item, "") if descriptions.get(item, "") else "")) for item in ids}
    role_names = sorted(existence.role.astype(str).unique().tolist()) if len(existence) else []
    prompts = ["task: sentence similarity | query: " + text_by_id[item] for item in ids]
    role_prompts = ["task: sentence similarity | query: " + role for role in role_names]
    with torch.inference_mode():
        vectors = encoder.encode(prompts, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
        raw_roles = encoder.encode(role_prompts, batch_size=8, normalize_embeddings=True, convert_to_numpy=True) if role_prompts else np.zeros((0,768), dtype=np.float32)
    del encoder
    if torch.cuda.is_available(): torch.cuda.empty_cache()
if OFFLINE_SMOKE:
    role_names = sorted(existence.role.astype(str).unique().tolist()) if len(existence) else []
    raw_roles = np.stack([smoke_vector(role) for role in role_names]) if role_names else np.zeros((0,768), dtype=np.float32)
vectors = np.asarray(vectors, dtype=np.float32)
raw_roles = np.asarray(raw_roles, dtype=np.float32)
if vectors.shape != (len(ids), 768) or not np.isfinite(vectors).all(): raise ValueError(f"Unexpected embedding shape/data: {vectors.shape}")
if raw_roles.shape != (len(role_names),768) or not np.isfinite(raw_roles).all(): raise ValueError(f"Unexpected role embedding shape/data: {raw_roles.shape}")
if np.any(np.linalg.norm(vectors, axis=1) < 0.98): raise ValueError("Expected normalized nonzero embeddings")
print({"embedding_shape": list(vectors.shape), "model_revision": loaded_revision, "device": DEVICE, "synthetic_smoke_only": OFFLINE_SMOKE})

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

node_index = {item: i for i, item in enumerate(ids)}
X = torch.tensor(vectors, dtype=torch.float32, device=DEVICE)
R = torch.tensor(raw_roles, dtype=torch.float32, device=DEVICE)

def project_ball(x, eps=1e-5):
    norm = torch.linalg.vector_norm(x, dim=-1, keepdim=True).clamp_min(1e-12)
    return x * torch.clamp((1.0-eps) / norm, max=1.0)
def hyp_dist(x, y):
    x, y = project_ball(x), project_ball(y)
    delta = ((x-y)**2).sum(dim=-1)
    dx = (1-(x*x).sum(dim=-1)).clamp_min(1e-6)
    dy = (1-(y*y).sum(dim=-1)).clamp_min(1e-6)
    return torch.acosh((1+2*delta/(dx*dy)).clamp_min(1+1e-6))
def depth(x):
    radius = torch.linalg.vector_norm(project_ball(x), dim=-1).clamp(max=1-1e-6)
    return 2*torch.atanh(radius)

class HierarchyAdapter(nn.Module):
    def __init__(self, input_dim=768, geometry_dim=64):
        super().__init__()
        if geometry_dim % 2: raise ValueError("geometry_dim must be even")
        self.direction = nn.Linear(input_dim, geometry_dim, bias=False)
        self.radial = nn.Linear(input_dim, 1)
        self.role_rotation_scale = nn.Linear(input_dim, 1 + geometry_dim//2)
    def forward(self, x):
        direction = F.normalize(self.direction(x), dim=-1, eps=1e-7)
        radius = 0.08 + 0.80*torch.sigmoid(self.radial(x))
        return project_ball(direction*radius)
    def existential(self, filler_points, role_inputs):
        params = self.role_rotation_scale(role_inputs)
        rotation = torch.tanh(params[...,1:])*torch.pi
        scale = torch.exp(0.25*torch.tanh(params[...,0:1]))
        left,right = filler_points.chunk(2, dim=-1)
        cosine,sine = torch.cos(rotation),torch.sin(rotation)
        rotated = torch.cat((left*cosine-right*sine,left*sine+right*cosine),dim=-1)
        return project_ball(rotated*scale)

model = HierarchyAdapter(768, GEOMETRY_DIM).to(DEVICE)
with torch.no_grad(): anchor = model(X).detach()
train_pairs = torch.tensor([[node_index[c], node_index[p]] for c,p in train_edges], dtype=torch.long, device=DEVICE)
role_index = {role: index for index, role in enumerate(role_names)}
role_trained = len(existence) >= 8
if role_trained:
    role_triplets = torch.tensor([[node_index[str(subject)], role_index[str(role)], node_index[str(filler)]] for subject,role,filler in existence[["subject","role","filler"]].itertuples(index=False,name=None)], dtype=torch.long, device=DEVICE)
else:
    role_triplets = None
    if len(existence): print(f"Role head disabled: {len(existence)} assertions available; require 8.")
val_index = {key: i for i,key in enumerate(ids)}
def mrr(edges, points):
    ranks = []
    with torch.no_grad():
        for child,parent in edges:
            ci,pi = val_index[child], val_index[parent]
            scores = -hyp_dist(points[ci].expand(len(ids),-1), points) + 0.8*(depth(points[ci])-depth(points))
            scores[ci] = -torch.inf
            ranks.append(1 + int((scores > scores[pi]).sum().item()))
    return {"examples":len(ranks), "mrr":float(np.mean([1/r for r in ranks])), "hits_at_1":float(np.mean([r<=1 for r in ranks])), "hits_at_3":float(np.mean([r<=3 for r in ranks])), "hits_at_5":float(np.mean([r<=5 for r in ranks])), "median_rank":float(np.median(ranks)), "ranks":ranks}
print({"adapter_parameters": sum(p.numel() for p in model.parameters()), "role_axioms": len(existence), "role_head_trained": role_trained})

In [ ]:
trainable = list(model.direction.parameters()) + list(model.radial.parameters()) + (list(model.role_rotation_scale.parameters()) if role_trained else [])
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.002)
best_mrr, best_state, best_epoch = -1.0, None, None
history = []
for epoch in range(1, EPOCHS+1):
    model.train(); optimizer.zero_grad(set_to_none=True)
    points = model(X)
    child, parent = points[train_pairs[:,0]], points[train_pairs[:,1]]
    loss = F.relu(hyp_dist(child,parent)-1.30).mean() + 2.0*F.relu(0.11+depth(parent)-depth(child)).mean() + 0.20*((points-anchor)**2).mean()
    if role_trained:
        role_source = points[role_triplets[:,0]]
        role_target = model.existential(points[role_triplets[:,2]], R[role_triplets[:,1]])
        loss = loss + 0.30*F.relu(hyp_dist(role_source,role_target)-1.5).mean()
    if not torch.isfinite(loss): raise FloatingPointError("Nonfinite hierarchy loss")
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        model.eval(); validation_metrics = mrr(val_edges, model(X).detach())
        history.append({"epoch":epoch, "loss":float(loss.detach().cpu()), "validation_mrr":validation_metrics["mrr"]})
        if validation_metrics["mrr"] > best_mrr:
            best_mrr, best_epoch = validation_metrics["mrr"], epoch
            best_state = {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
if best_state is None: raise RuntimeError("No validation checkpoint produced")
model.load_state_dict(best_state); model.eval()
with torch.no_grad(): learned = model(X).detach()
test_metrics = mrr(test_edges, learned)
baseline_ranks = []
for child,parent in test_edges:
    ci,pi = node_index[child], node_index[parent]
    scores = X @ X[ci]; scores[ci] = -torch.inf
    baseline_ranks.append(1 + int((scores > scores[pi]).sum().item()))
baseline_metrics = {"examples":len(baseline_ranks), "mrr":float(np.mean([1/r for r in baseline_ranks])), "hits_at_1":float(np.mean([r<=1 for r in baseline_ranks])), "hits_at_3":float(np.mean([r<=3 for r in baseline_ranks])), "hits_at_5":float(np.mean([r<=5 for r in baseline_ranks])), "median_rank":float(np.median(baseline_ranks)), "ranks":baseline_ranks}
metrics = {"frozen_text_cosine":baseline_metrics, "poincare_adapter":test_metrics, "validation_mrr":best_mrr, "best_epoch":best_epoch}
if learned.shape != (len(ids),GEOMETRY_DIM) or not torch.isfinite(learned).all() or float(learned.norm(dim=1).max()) >= 1: raise ValueError("Invalid geometry vectors")
print(json.dumps(metrics, indent=2))
print("ADVISORY_ONLY: ranked unknown candidates are not false labels, and these scores are not OWL entailment.")

## Export isolated experiment artifacts

The adapter and 64-dimensional geometry vectors are experimental artifacts, not `semantic_768`, Qdrant payloads, canonical tuples, or ACE context. The receipt explicitly records caller-supplied data as unverified and production admission as false.

In [ ]:
vectors_path = RUN_DIR/"ontology_geometry_64.npz"
np.savez_compressed(vectors_path, ids=np.asarray(ids,dtype=str), vectors=learned.detach().cpu().numpy().astype(np.float32))
text_vectors_path = RUN_DIR/"frozen_text_embeddings_768_EXPERIMENT.npz"
np.savez_compressed(text_vectors_path, ids=np.asarray(ids,dtype=str), vectors=vectors.astype(np.float32))
role_vectors_path = RUN_DIR/"frozen_role_embeddings_768_EXPERIMENT.npz" if role_names else None
if role_vectors_path is not None: np.savez_compressed(role_vectors_path, ids=np.asarray(role_names,dtype=str), vectors=raw_roles.astype(np.float32))
adapter_path = RUN_DIR/"hierarchy_adapter.pt"
adapter_state = {key:value for key,value in best_state.items() if not key.startswith("role_rotation_scale.")}
role_state = {key:value for key,value in best_state.items() if key.startswith("role_rotation_scale.")} if role_trained else None
torch.save({"hierarchy_adapter_state":adapter_state,"role_adapter_state":role_state,"embedding_input_dim":768,"geometry_dim":GEOMETRY_DIM,"best_epoch":best_epoch}, adapter_path)
split_records = {name:[{"child":child,"parent":parent,"source_ids":edge_sources.get((child,parent),[])} for child,parent in split] for name,split in [("train",train_edges),("validation",val_edges),("test",test_edges)]}
split_path = RUN_DIR/"edge_splits.json"; split_path.write_text(json.dumps(split_records,indent=2,sort_keys=True)+"\n",encoding="utf-8")
history_path = RUN_DIR/"training_history.json"; history_path.write_text(json.dumps(history,indent=2,sort_keys=True)+"\n",encoding="utf-8")
metrics_path = RUN_DIR/"ranking_metrics.json"; metrics_path.write_text(json.dumps(metrics,indent=2,sort_keys=True)+"\n",encoding="utf-8")
import importlib.metadata
package_versions = {name:importlib.metadata.version(name) for name in ("torch","sentence-transformers","transformers","huggingface-hub","rdflib","pandas")}
manifest = {"schema":"parent-atlas.embeddinggemma-poincare-experiment.v1","created_at":datetime.now(timezone.utc).isoformat(),"status":"SMOKE_ONLY" if OFFLINE_SMOKE else "ADVISORY_ONLY","production_admission":False,"source_authority":"SYNTHETIC_SMOKE" if SOURCE_MODE == "demo" else "CALLER_SUPPLIED_UNVERIFIED","independent_source_grant_verified":False,"workspace_revision":None,"workspace_revision_status":"NOT_PROVIDED","input_artifacts":INPUT_ARTIFACTS,"input_sha256":INPUT_SHA256,"dataset_sha256":DATASET_SHA256,"input_mode":SOURCE_MODE,"verified_flags_are_caller_assertions":SOURCE_MODE == "csv","model_id":"SYNTHETIC_FAKE_NO_MODEL" if OFFLINE_SMOKE else MODEL_ID,"model_revision":loaded_revision,"requested_model_revision":MODEL_REVISION if not OFFLINE_SMOKE else "NOT_LOADED_SYNTHETIC","tokenizer_revision":MODEL_REVISION if not OFFLINE_SMOKE else "NOT_LOADED_SYNTHETIC","encoder_frozen":True,"encoder_dimension":int(vectors.shape[1]),"geometry_dimension":GEOMETRY_DIM,"embedding_recipe":"task: sentence similarity | query: {label and description}; roles use same symmetric recipe","backbone_dtype":str(DTYPE),"adapter_dtype":"torch.float32","packages":package_versions,"seed":SEED,"classes":len(ids),"asserted_edges":len(asserted),"edge_splits":{name:len(records) for name,records in split_records.items()},"train_closure_has_heldout_edges":False,"best_epoch":best_epoch,"metrics":metrics,"role_assertion_count":len(existence),"role_head_trained":role_trained,"role_evaluation":"NOT_RUN" if role_trained else "NOT_AVAILABLE","full_owl_reasoner":"NOT_RUN","transductive_evaluation":True,"unseen_class_generalization":"NOT_TESTED","unknown_candidates_are_false":False,"canonical_store_writes":False}
manifest_path = RUN_DIR/"experiment_manifest.json"; manifest_path.write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\n",encoding="utf-8")
artifacts = [vectors_path,text_vectors_path,adapter_path,split_path,history_path,metrics_path,manifest_path] + ([role_vectors_path] if role_vectors_path is not None else [])
checksums = {path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in artifacts}
checksum_path = RUN_DIR/"sha256.json"; checksum_path.write_text(json.dumps(checksums,indent=2,sort_keys=True)+"\n",encoding="utf-8")
archive = RUN_DIR.parent/f"parent_atlas_{MODEL_ID.rsplit('/',1)[-1]}_poincare_{DATASET_SHA256[:12]}.zip"
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in artifacts + [checksum_path]: bundle.write(path,path.name)
print(json.dumps(manifest,indent=2)); files.download(str(archive))